In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
import pandas as pd
import backbone as bb
import pickle
from tensorflow.keras import layers
from tensorflow.keras import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import SGD,Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.layers import Flatten,Dense,BatchNormalization,Activation,Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
# from tensorflow.keras.preprocessing import flow_from_dataframe

In [2]:
#import train labels dataset using backbone module
file_path_trainlabels, file_path_trainimages, train_labels = bb.init_data()

train_labels['label'] = train_labels['label'].astype('str')

In [3]:
IMG_SIZE = (224,224) # this image size depends on what the model we are using requires! mobilenet wants 224x224 in any case

### Preprocessing data, generating training and validation split

In [4]:
#Data augmentation for the training set:
train_datagen = ImageDataGenerator(#rescale=1.0/255,
                                   validation_split=0.2,
                                    rotation_range=2, 
                                    horizontal_flip=True,
                                    zoom_range=0.1)

#Pre-processing the validation set (without data augmentation):
val_datagen = ImageDataGenerator(validation_split=0.2)

#Initiate a generator for the test set
test_datagen = ImageDataGenerator() #rescale=1./255

In [5]:
#Train dataset generator
train_generator=train_datagen.flow_from_dataframe(dataframe=train_labels, #dataframe with the image names and labels
                                            directory=file_path_trainimages, #filepath images
                                            x_col='img_name',
                                            y_col='label',
                                            subset='training',
                                            color_mode="rgb",
                                            batch_size=32,
                                            seed=42, #how the data is randomized
                                            shuffle=True,
                                            class_mode='categorical',
                                            class_names=None,
                                            interpolation='nearest', #interpolation for resizing images. 
                                            target_size=IMG_SIZE)
#Efficientnet uses bicubic interpolation to resize images so perhaps use that too? - No improvement

Found 24490 validated image filenames belonging to 80 classes.


In [10]:
#Validation dataset generator
val_generator=val_datagen.flow_from_dataframe(dataframe=train_labels,
                                            directory=file_path_trainimages,
                                            x_col='img_name',
                                            y_col='label',
                                            subset='validation',
                                            color_mode="rgb",
                                            batch_size=32,
                                            seed=42,
                                            shuffle=True,
                                            class_mode='categorical',
                                            class_names=None,
                                            interpolation='nearest',
                                            target_size=IMG_SIZE)

Found 6122 validated image filenames belonging to 80 classes.


### Generating class weights

In [11]:
#Balancing dataset by setting weights to the labels
from sklearn.utils import class_weight

class_weights = class_weight.compute_class_weight('balanced', np.unique(train_generator.classes), train_generator.classes)

class_weights = {i : class_weights[i] for i in train_generator.classes}

#then use:    model.fit_generator(train_generator, class_weight=class_weights)

## Creating the base model (Efficientnet B0)

Efficientnet B0 is pre-trained on ImageNet (1.4 images, 1000 classes).

First we need to pick which layer of MobileNetv2 we will use for feature extraction. The very last classification layer (i.e. the 'top' layer) is not very useful since these features might be too specific.

Instead we will use the common practice to depend on the very last layer before the flatten operation, also called the 'bottleneck layer'. The bottleneck layer features retain more generality as compared to the final/top layer.

We instantiate an Efficientnet B0 model pre-loaded with the weights trained on Imagenet. By speficying include_top = False, we load a network that doesn't include the classification layers at the top, which is ideal for feature extraction.

In [12]:
# create base model from pre-trained mobilenet v2
IMG_SHAPE = (IMG_SIZE) + (3,)
base_model = tf.keras.applications.EfficientNetB0(input_shape = IMG_SHAPE,
                                              include_top = False,
                                              weights = 'imagenet')

In [13]:
IMG_SHAPE

(224, 224, 3)

## Feature Extraction

In [14]:
base_model.trainable = False # we prevent the weights from being updated during training

# taking a look at our base model architecture:
base_model.summary()

Model: "efficientnetb0"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 224, 224, 3) 0                                            
__________________________________________________________________________________________________
rescaling (Rescaling)           (None, 224, 224, 3)  0           input_1[0][0]                    
__________________________________________________________________________________________________
normalization (Normalization)   (None, 224, 224, 3)  7           rescaling[0][0]                  
__________________________________________________________________________________________________
stem_conv_pad (ZeroPadding2D)   (None, 225, 225, 3)  0           normalization[0][0]              
_____________________________________________________________________________________

### Building the model

In [15]:
inputs = tf.keras.Input(shape=(IMG_SHAPE))

x = base_model(inputs, training=False)
x = layers.BatchNormalization()(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(80, activation='softmax')(x)
model = Model(inputs, outputs)
model.summary()

Model: "functional_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_2 (InputLayer)         [(None, 224, 224, 3)]     0         
_________________________________________________________________
efficientnetb0 (Functional)  (None, 7, 7, 1280)        4049571   
_________________________________________________________________
batch_normalization (BatchNo (None, 7, 7, 1280)        5120      
_________________________________________________________________
global_average_pooling2d (Gl (None, 1280)              0         
_________________________________________________________________
dropout (Dropout)            (None, 1280)              0         
_________________________________________________________________
dense (Dense)                (None, 80)                102480    
Total params: 4,157,171
Trainable params: 105,040
Non-trainable params: 4,052,131
______________________________________

### Training the model

In [ ]:
model.compile(optimizer=Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.fit(x=train_generator,validation_data=val_generator,epochs=2,verbose=1,
                    class_weight=class_weights)

base_model.trainable = True
model.compile(optimizer=Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.fit(x=train_generator,validation_data=val_generator,epochs=1,verbose=1,
                    class_weight=class_weights)

base_model.trainable = False
model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.fit(x=train_generator,validation_data=val_generator,epochs=2,verbose=1,
                    class_weight=class_weights)

base_model.trainable = True
model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.fit(x=train_generator,validation_data=val_generator,epochs=1,verbose=1,
                    class_weight=class_weights)

base_model.trainable = False
model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.fit(x=train_generator,validation_data=val_generator,epochs=2,verbose=1,
                    class_weight=class_weights)

base_model.trainable = True
model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.fit(x=train_generator,validation_data=val_generator,epochs=1,verbose=1,
                    class_weight=class_weights)

### Saving the model

In [ ]:
#save model using built-in function
model.save('efficientnet_b0_model1_tim_retraining.h5')
model.save_weights('efficientnet_b0_model1_tim_weights.h5')

### Generating the learning curves

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

plt.figure(figsize=(8, 8))
# plt.subplot(2, 1, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([min(plt.ylim()),1])
plt.title('Training and Validation Accuracy')


## Classifying the Test Dataset

In [ ]:
file_path_testimages, file_path_testlabels, test_labels = bb.get_test_paths()

#Test dataset generator
test_generator=test_datagen.flow_from_dataframe(dataframe=test_labels,
                                            directory=file_path_testimages,
                                            x_col='img_name',
                                            y_col='label',
                                            color_mode="rgb",
                                            batch_size=32,
                                            seed=42,
                                            shuffle=False,
                                            class_mode=None,
                                            interpolation='nearest',
                                            target_size=IMG_SIZE)

In [ ]:
#load model if required
#from tensorflow.keras.models import load_model
#model = load_model('efficientnet_b0_model2.h5')

In [ ]:
test_generator.reset()

#classify our test data
classify = model.predict(test_generator,
                         verbose=1)

In [ ]:
predictions = np.argmax(classify, axis=1)
predictions

In [ ]:
#map predicted labels to the labels indices
sorted_labels = (train_generator.class_indices)
sorted_labels = dict((v,k) for k,v in sorted_labels.items())
predicted_labels = [sorted_labels[k] for k in predictions]

In [ ]:
#save output
submission_file = pd.DataFrame()
submission_file['img_name'] = test_labels['img_name']
submission_file['label'] = predicted_labels
submission_file.to_csv('test_labels_predictions2.csv', index=False)

In [ ]:
#check prediction
plt.figure(figsize=(10,10))

test_generator.reset()
sample = next(test_generator)

for item in range(9):
    ax = plt.subplot(3, 3, item + 1)
    img = sample[item].astype('uint8')
    label = predicted_labels[item]
    plt.axis('off')
    plt.imshow(img)
    plt.title(label)